# Demo: flujo de delegación del Orquestador Multi-Agente

Corre el escenario completo del orquestador (Supervisor + Agente de
Investigación + Agente de Análisis) paso a paso, mostrando en cada celda
a quién delega el Supervisor, qué hace cada especialista, y por qué el
Supervisor decide refinar o cerrar.

Usa los mismos LLMs **determinísticos** de prueba que `demo_offline.py`
(sin ninguna API key, sin red) — así este notebook se puede correr y
revisar tal cual, sin depender de credenciales. El grafo, el `StateGraph`,
el `ToolNode` interno de cada sub-agente ReAct y las tres herramientas que
se ven acá son los reales; lo único "de utilería" es qué herramienta
decide llamar el LLM en cada paso. Para una corrida con razonamiento real,
ver `run_demo.py`.

In [1]:
from demo_offline import PREGUNTA, _armar_guiones
from graph import construir_grafo
from state import estado_inicial

print("Consulta del usuario:")
print(PREGUNTA)

Consulta del usuario:
Investigá las opiniones sobre el lanzamiento del auricular Aurora X2, analizá el sentimiento general y el promedio de las calificaciones, y dame un resumen.


## Armamos el grafo

Cada rol (supervisor, investigador, analista) recibe su propio LLM
determinístico inyectado — el mismo mecanismo de `construir_grafo(obtener_llm_...)`
que se usa en los tests offline (`tests/test_graph_ruteo.py`).

In [2]:
supervisor_fake, investigador_fake, analista_fake = _armar_guiones()

grafo = construir_grafo(
    obtener_llm_supervisor=lambda: supervisor_fake,
    obtener_llm_investigador=lambda: investigador_fake,
    obtener_llm_analista=lambda: analista_fake,
).compile()

print(grafo.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Corremos el grafo paso a paso (`astream`)

En vez de invocar todo de una (`ainvoke`), usamos `astream(..., stream_mode="values")`
para poder imprimir, después de cada nodo, el estado completo tal como
queda en ese instante — así se ve la delegación en vivo: quién actuó, qué
aportó, y a quién le toca ahora.

In [3]:
def mostrar_paso(paso: int, estado: dict) -> None:
    ultimo_mensaje = estado["messages"][-1]
    quien = getattr(ultimo_mensaje, "name", None) or ultimo_mensaje.__class__.__name__
    contenido = str(ultimo_mensaje.content).strip() or "(tool call interno, ver detalle abajo)"
    print(f"--- Paso {paso}: [{quien}] " + "-" * 40)
    print(contenido[:400])
    print(f"    next_agent={estado['next_agent']!r}  tarea_completada={estado['tarea_completada']}  "
          f"pasos_dados={estado['pasos_dados']}  contribuciones_acumuladas={len(estado['contribuciones'])}")
    print()


estado_final = None
paso = 0
async for estado in grafo.astream(
    estado_inicial(PREGUNTA), config={"recursion_limit": 15}, stream_mode="values"
):
    mostrar_paso(paso, estado)
    estado_final = estado
    paso += 1

--- Paso 0: [HumanMessage] ----------------------------------------
Investigá las opiniones sobre el lanzamiento del auricular Aurora X2, analizá el sentimiento general y el promedio de las calificaciones, y dame un resumen.
    next_agent=None  tarea_completada=False  pasos_dados=0  contribuciones_acumuladas=0

--- Paso 1: [supervisor] ----------------------------------------
[Supervisor -> investigador] Todavía no hay ninguna contribución: hay que investigar primero.
    next_agent='investigador'  tarea_completada=False  pasos_dados=1  contribuciones_acumuladas=1

--- Paso 2: [investigador] ----------------------------------------
Encontré 5 reseñas sobre el lanzamiento del auricular Aurora X2:
- (5/5, TiendaApp Reviews): El nuevo auricular Aurora X2 tiene un sonido excelente y la batería dura muchísimo más que el modelo anterior. Totalmente recomendado.
- (3/5, TiendaApp Reviews): Buen producto pero la app para configurarlo se cuelga seguido. El audio en sí es muy bueno.
- (2/5, For

## Estado final: contribuciones estructuradas

Esto es el "Estado Compartido Estructurado" que pide el enunciado: cada
aporte queda registrado con quién lo hizo, en orden — se puede reconstruir
exactamente el camino de delegación (incluido el refinamiento del
Analista) sin tener que releer/parsear todo el historial de mensajes.

In [4]:
for i, contribucion in enumerate(estado_final["contribuciones"], start=1):
    print(f"{i}. [{contribucion['agente']}] {contribucion['contenido']}")

1. [supervisor] Buscá reseñas y opiniones sobre el lanzamiento del auricular Aurora X2.
2. [investigador] Encontré 5 reseñas sobre el lanzamiento del auricular Aurora X2:
- (5/5, TiendaApp Reviews): El nuevo auricular Aurora X2 tiene un sonido excelente y la batería dura muchísimo más que el modelo anterior. Totalmente recomendado.
- (3/5, TiendaApp Reviews): Buen producto pero la app para configurarlo se cuelga seguido. El audio en sí es muy bueno.
- (2/5, Foro TecnoLatam): Decepcionante. Prometían cancelación de ruido de nivel premium y en la práctica es mediocre, igual que la generación pasada.
- (5/5, TiendaApp Reviews): Relación precio-calidad increíble. Lo compré para el gimnasio y no se cae ni se corta la conexión bluetooth.
- (4/5, Blog GadgetsHoy): Un lanzamiento sólido, con mejoras reales sobre la versión anterior, aunque el precio quedó un poco por encima de la competencia directa.
3. [supervisor] Con las reseñas que encontró el investigador, calculá el sentimiento general Y

## Resumen

- Pasos usados por el Supervisor vs. el límite configurado
  (`MAX_PASOS_SUPERVISOR`, ver `config.py`).
- Si la tarea se marcó como completa.
- La síntesis final que el Supervisor le devuelve al usuario.

In [5]:
from config import MAX_PASOS_SUPERVISOR

print(f"Pasos del supervisor: {estado_final['pasos_dados']} / {MAX_PASOS_SUPERVISOR} máx.")
print(f"Tarea completada: {estado_final['tarea_completada']}")
print()
print("Síntesis final del Supervisor:")
print(estado_final["messages"][-1].content)

Pasos del supervisor: 4 / 6 máx.
Tarea completada: True

Síntesis final del Supervisor:
Resumen final: las opiniones sobre el Aurora X2 son, en general, positivas (sentimiento positivo, score 0.47), con un promedio de calificación de 3.8/5 sobre 5 reseñas.
